# backprop-pop-outgrad-loop — worked example 3: Reverse pass through a sum-reduction (scalar loss back to a vector leaf)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backprop-pop-outgrad-loop`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A reduction like `s = sum(v)` collapses a vector leaf to a scalar. Its back_fn must **broadcast** the scalar upstream grad back to the leaf's shape: `dL/dv = grad_out * ones_like(v)`. The driver is unchanged — pop the scalar's grad, dispatch the sum back_fn, accumulate the broadcast result into the leaf's slot. This shows the loop is shape-agnostic; correctness lives in the back_fns.

## Worked solution

We compute `s = sum(v)` where `v` is a length-5 leaf, then run the reverse pass. Analytically every element of `v` contributes 1 to `s`, so `v.grad` should be all-ones (scaled by the seed).

**Step 1 — seed.** `s` is a scalar; `end_grad = ones_like(s.array)` (a scalar `1.0`). `grads = {id(s): 1.0}`.

**Step 2 — pop `s`.** `s` is a `sum` node with one parent `v` at arg 0. The sum back_fn takes the scalar `grad_out` and broadcasts it to `v`'s shape: `grad_out * t.ones_like(x)` where `x` is the recorded forward arg (the value of `v`). This yields a length-5 vector of the scalar value. We accumulate it into `grads[id(v)]`.

**Step 3 — pop `v`.** Leaf, so write the broadcast grad into `v.grad`.

**Why it works.** The driver never inspects shapes — it blindly pops, dispatches, and accumulates. The shape-correctness (scalar → vector) is entirely the back_fn's job via broadcasting. We cross-check with autograd: build `v_t` with `requires_grad=True`, do `v_t.sum().backward()`, and compare `v_t.grad` to our `v.grad`.

In [ ]:
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _sum_back0(go, out, x):  return go * t.ones_like(x)

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

t.manual_seed(0)
v = MiniTensor(t.randn(5))
s_arr = v.array.sum()
s = MiniTensor(s_arr, Recipe('sum', (v.array,), {}, {0: v}))
back_funcs = {('sum', 0): _sum_back0}
backprop(s, t.ones_like(s.array), [s, v], back_funcs)

v_t = v.array.clone().requires_grad_(True)
v_t.sum().backward()
print('matches autograd:', t.allclose(v.grad, v_t.grad))
print('v.grad:', v.grad)